# CS781 Phase 5 - Legal Statute Identification - notebook `4` (the study)

**Base design:** document text in Phase-1 tokenization -> **the whole document up to 2,048 tokens** (4 chunks of 512;
longer documents: 4 chunks spread evenly over it) -> Phase-1 backbone -> mean-pooled chunk vectors + relative chunk
position (16 sinusoids) -> **attention pooling** -> **one sigmoid per statute** (binary relevance), BCE loss; 1 epoch over
train.json, batches of 8 documents; checkpoint chosen on a fixed 3,000-document dev subset (macro-F1 at the best global
threshold), evaluated every 25% of the steps; the final report uses all of dev with three decision rules (0.5, best
global threshold, per-statute thresholds + an honest cross-fitted estimate); every document gets at least one statute.
Training documents that also occur in dev/test are removed.
**Techniques** (T, every model): LoRMA (compulsory), AdaLoRA, QLoRA, adapters, prefix tuning, partial FT.
**Study** (S, models B and D, LoRMA, the same 12,000-document subset): which part of the document, head (label-wise
attention, chunk-level predictions + max, mean pooling), loss (BCE + positive weights, ASL, focal), statute co-occurrence.
**Extras** (X, run last): full FT, prompt tuning. Knowledge distillation is optional (code ready, not scheduled).

| ID | GPU | run | technique | change vs base design | group | estimate |
|---|---|---|---|---|---|---|
| D14 | GPU0 | `D14_s_spread8` | LoRMA r16 | whole document up to 4,096 tokens (8 chunks) | S | ~75 min |
| D16 | GPU0 | `D16_s_chunkmax` | LoRMA r16 | chunk predictions + max aggregation | S | ~56 min |
| D18 | GPU0 | `D18_s_bcepw` | LoRMA r16 | BCE + positive weights | S | ~56 min |
| D20 | GPU0 | `D20_s_focal` | LoRMA r16 | focal loss | S | ~56 min |
| B14 | GPU0 | `B14_s_spread8` | LoRMA r16 | whole document up to 4,096 tokens (8 chunks) | S | ~54 min |
| D12 | GPU0 | `D12_s_first1` | LoRMA r16 | first 512 tokens only | S | ~44 min |
| B15 | GPU0 | `B15_s_lwan` | LoRMA r16 | label-wise attention head | S | ~40 min |
| B17 | GPU0 | `B17_s_mean` | LoRMA r16 | mean pooling of chunks | S | ~40 min |
| B19 | GPU0 | `B19_s_asl` | LoRMA r16 | asymmetric loss (ASL) | S | ~40 min |
| B21 | GPU0 | `B21_s_cooc` | LoRMA r16 | statute co-occurrence layer | S | ~40 min |
| B12 | GPU0 | `B12_s_first1` | LoRMA r16 | first 512 tokens only | S | ~31 min |
| D11 | GPU1 | `D11_s_ref` | LoRMA r16 | 12k-doc subset (reference) | S | ~56 min |
| D15 | GPU1 | `D15_s_lwan` | LoRMA r16 | label-wise attention head | S | ~56 min |
| D17 | GPU1 | `D17_s_mean` | LoRMA r16 | mean pooling of chunks | S | ~56 min |
| D19 | GPU1 | `D19_s_asl` | LoRMA r16 | asymmetric loss (ASL) | S | ~56 min |
| D21 | GPU1 | `D21_s_cooc` | LoRMA r16 | statute co-occurrence layer | S | ~56 min |
| D13 | GPU1 | `D13_s_last2` | LoRMA r16 | last 1,024 tokens | S | ~50 min |
| B11 | GPU1 | `B11_s_ref` | LoRMA r16 | 12k-doc subset (reference) | S | ~40 min |
| B16 | GPU1 | `B16_s_chunkmax` | LoRMA r16 | chunk predictions + max aggregation | S | ~40 min |
| B18 | GPU1 | `B18_s_bcepw` | LoRMA r16 | BCE + positive weights | S | ~40 min |
| B20 | GPU1 | `B20_s_focal` | LoRMA r16 | focal loss | S | ~40 min |
| B13 | GPU1 | `B13_s_last2` | LoRMA r16 | last 1,024 tokens | S | ~36 min |

**Inputs:** dataset `cs781-p5-code`, `cs781-legal-data` (tokenizer), Phase-1 checkpoint(s) of model B + D.
Internet ON, secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
Expected: ~8.9 h of runs (GPU0 8.9 h, GPU1 8.8 h) after ~5 min of tokenization and a
~5 min smoke test. Each run has a time limit (it then stops, is evaluated on dev and kept); no run starts after
11.3 h, so the notebook always finishes inside Kaggle's 12-hour limit. A run that fails the smoke test is dropped
(its error is in the report); the others continue.
**Outputs:** `submissions/<run>.zip` (ready to submit), `models/<run>.pt`, `bundle_p5_4.zip` (dev/test probabilities,
attention weights, logs; input of `cs781-p5-final`), `REPORT_p5_4.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
NB_START = time.time()
W = "/kaggle/working"
WORK, P5DATA, RUNS, MODELS = f"{W}/p5code", "/tmp/p5data", f"{W}/runs", f"{W}/models"
CACHE = "/tmp/p5_cache.npz"                        # tokenized data (outside /kaggle/working: not part of the output)
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions", P5DATA):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("lsi_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-5 data from Hugging Face (Phase_5_LSI/Train/{train,dev}.json, Phase_5_LSI/Test/test.json)
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
files = [f for f in list_repo_files(REPO, repo_type="dataset", token=tok_) if f.startswith("Phase_5")]
print("HF files:", files)
for f in files:
    dst = os.path.join(P5DATA, f)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), dst)

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
ALL_PT = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
print("checkpoint files found under /kaggle/input:", *ALL_PT, sep="\n  ")
for p in ALL_PT:
    if os.path.basename(p) != "final.pt" or "smoke" in p.lower():
        continue
    m = re.search(r"run[_-]([a-e])(?:[_/-]|$)", p, re.I) or re.search(r"model[_-]?([a-e])(?:[_/-]|$)", p, re.I)
    if not m:
        continue
    k = m.group(1).lower()
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
CKPT.update(CKPT_OVERRIDE if "CKPT_OVERRIDE" in globals() else {})   # e.g. CKPT_OVERRIDE = {"c": "/kaggle/input/.../final.pt"}
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- unit tests, then tokenize every split once (shared by all runs; several CPU processes, a few minutes)
r = subprocess.run("python tests_lsi.py", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"
import lsi_data
t0 = time.time()
INFO = lsi_data.build_cache(P5DATA, TOK_ROOT, CACHE)
print(f"cache built in {time.time() - t0:.0f}s")
if "test" not in INFO:
    print("WARNING: no test.json found - runs will train and save models, but write no submission files")

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'lsi_data.py', 'lsi_eval.py', 'lsi_model.py', 'lsi_train.py', 'models.py', 'ner_data.py', 'ner_model.py', 'p1_loader.py', 'peft.py', 'tests_lsi.py']
HF files: ['Phase_5_LSI/Test/test.json', 'Phase_5_LSI/Train/dev.json', 'Phase_5_LSI/Train/train.json']


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

checkpoint files found under /kaggle/input:
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/latest.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/latest.pt
  model b: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/final.pt
  model d: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn

In [2]:
NB_HOURS = 11.5                                    # Kaggle stops a session at 12 h; no run starts after this

def cmd(e, out, extra, model_dir):
    return (f"python lsi_train.py --ckpt {CKPT[e['model']]} --tok_root {TOK_ROOT} --data_dir {P5DATA} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {e['args']} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    return p.returncode

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel. smoke: a few documents, a few steps."""
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            if smoke:
                extra = "--max_docs 24 --batch_docs 4 --eval_every 0.5" + ("" if e is jobs[0] else " --no_test")
            else:
                left = NB_HOURS * 60 - (time.time() - NB_START) / 60
                if left < 20:
                    print(f"[GPU{gpu}] {t}: SKIPPED (notebook time budget used up)", flush=True)
                    status[t] = "skipped"
                    continue
                extra = f"--max_minutes {min(e['max_min'], left - 15):.0f}"
            try:
                status[t] = run(t, cmd(e, f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
            except Exception as ex:                       # e.g. a missing checkpoint: report it, keep the queue going
                print(f"[GPU{gpu}] {t}: NOT STARTED - {ex!r}", flush=True)
                open(f"{W}/logs/{t}.txt", "w").write(f"not started: {ex!r}")
                status[t] = "error"
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Every experiment once on 24 documents. A failing experiment is dropped (its error is printed and reported)
    instead of stopping the notebook; the notebook stops only if every experiment fails."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    return [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]

## Experiments (one queue per GPU)

In [3]:
NAME = '4'
QUEUES = [[{'id': 'D14',
   'tag': 'D14_s_spread8',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'whole document up to 4,096 tokens (8 chunks)',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --n_chunks 8',
   'max_min': 125},
  {'id': 'D16',
   'tag': 'D16_s_chunkmax',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'chunk predictions + max aggregation',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --head chunkmax',
   'max_min': 95},
  {'id': 'D18',
   'tag': 'D18_s_bcepw',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'BCE + positive weights',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --loss bce_pw',
   'max_min': 95},
  {'id': 'D20',
   'tag': 'D20_s_focal',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'focal loss',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --loss focal',
   'max_min': 95},
  {'id': 'B14',
   'tag': 'B14_s_spread8',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'whole document up to 4,096 tokens (8 chunks)',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --n_chunks 8',
   'max_min': 91},
  {'id': 'D12',
   'tag': 'D12_s_first1',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'first 512 tokens only',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --select first '
           '--n_chunks 1',
   'max_min': 75},
  {'id': 'B15',
   'tag': 'B15_s_lwan',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'label-wise attention head',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --head lwan',
   'max_min': 70},
  {'id': 'B17',
   'tag': 'B17_s_mean',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'mean pooling of chunks',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --head mean',
   'max_min': 70},
  {'id': 'B19',
   'tag': 'B19_s_asl',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'asymmetric loss (ASL)',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --loss asl',
   'max_min': 70},
  {'id': 'B21',
   'tag': 'B21_s_cooc',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'statute co-occurrence layer',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --cooc',
   'max_min': 70},
  {'id': 'B12',
   'tag': 'B12_s_first1',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'first 512 tokens only',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --select first '
           '--n_chunks 1',
   'max_min': 55}],
 [{'id': 'D11',
   'tag': 'D11_s_ref',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': '12k-doc subset (reference)',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000',
   'max_min': 95},
  {'id': 'D15',
   'tag': 'D15_s_lwan',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'label-wise attention head',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --head lwan',
   'max_min': 95},
  {'id': 'D17',
   'tag': 'D17_s_mean',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'mean pooling of chunks',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --head mean',
   'max_min': 95},
  {'id': 'D19',
   'tag': 'D19_s_asl',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'asymmetric loss (ASL)',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --loss asl',
   'max_min': 95},
  {'id': 'D21',
   'tag': 'D21_s_cooc',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'statute co-occurrence layer',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --cooc',
   'max_min': 95},
  {'id': 'D13',
   'tag': 'D13_s_last2',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'last 1,024 tokens',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --select last '
           '--n_chunks 2',
   'max_min': 85},
  {'id': 'B11',
   'tag': 'B11_s_ref',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': '12k-doc subset (reference)',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000',
   'max_min': 70},
  {'id': 'B16',
   'tag': 'B16_s_chunkmax',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'chunk predictions + max aggregation',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --head chunkmax',
   'max_min': 70},
  {'id': 'B18',
   'tag': 'B18_s_bcepw',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'BCE + positive weights',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --loss bce_pw',
   'max_min': 70},
  {'id': 'B20',
   'tag': 'B20_s_focal',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'focal loss',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --loss focal',
   'max_min': 70},
  {'id': 'B13',
   'tag': 'B13_s_last2',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'last 1,024 tokens',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 12000 --select last '
           '--n_chunks 2',
   'max_min': 63}]]
ALL_QUEUES = [list(q) for q in QUEUES]                    # the full plan (the report lists dropped runs too)

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU1] smoke_D11_s_ref: exit 0 | [20:18:35] TEST: 24 documents, 55.58 statutes each (threshold 0.33) -> submission.zip / [20:18:35] DONE
[GPU0] smoke_D14_s_spread8: exit 0 | [20:18:36] TEST: 24 documents, 98.75 statutes each (threshold 0.24) -> submission.zip / [20:18:36] DONE
[GPU1] smoke_D15_s_lwan: exit 0 | [20:19:00] TEST: no test.json - no submission written / [20:19:00] DONE
[GPU0] smoke_D16_s_chunkmax: exit 0 | [20:19:00] TEST: no test.json - no submission written / [20:19:00] DONE
[GPU1] smoke_D17_s_mean: exit 0 | [20:19:18] TEST: no test.json - no submission written / [20:19:18] DONE
[GPU0] smoke_D18_s_bcepw: exit 0 | [20:19:18] TEST: no test.json - no submission written / [20:19:18] DONE
[GPU1] smoke_D19_s_asl: exit 0 | [20:19:37] TEST: no test.json - no submission written / [20:19:37] DONE
[GPU0] smoke_D20_s_focal: exit 0 | [20:19:37] TEST: no test.json - no submission written / [20:19:37] DONE
[GPU1] smoke_D21_s_cooc: exit 0 | [20:19:55] TEST: no test.json - no submission w

## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU1] D11_s_ref: exit 0 | [21:24:38] TEST: 13019 documents, 8.60 statutes each (threshold 0.10) -> submission.zip / [21:24:38] DONE
[GPU0] D14_s_spread8: exit 0 | [21:45:38] TEST: 13019 documents, 7.35 statutes each (threshold 0.12) -> submission.zip / [21:45:38] DONE
[GPU1] D15_s_lwan: exit 0 | [22:28:37] TEST: 13019 documents, 7.66 statutes each (threshold 0.11) -> submission.zip / [22:28:37] DONE
[GPU0] D16_s_chunkmax: exit 0 | [22:48:44] TEST: 13019 documents, 7.17 statutes each (threshold 0.12) -> submission.zip / [22:48:44] DONE
[GPU1] D17_s_mean: exit 0 | [23:32:38] TEST: 13019 documents, 7.91 statutes each (threshold 0.11) -> submission.zip / [23:32:38] DONE
[GPU0] D18_s_bcepw: exit 0 | [23:51:56] TEST: 13019 documents, 5.40 statutes each (threshold 0.45) -> submission.zip / [23:51:56] DONE
[GPU1] D19_s_asl: exit 0 | [00:36:27] TEST: 13019 documents, 7.23 statutes each (threshold 0.50) -> submission.zip / [00:36:27] DONE
[GPU0] D20_s_focal: exit 0 | [00:55:00] TEST: 13019 docu

## 3. Report

In [6]:
def log_tail(name, n=2):
    p = f"{W}/logs/{name}.txt"
    return " / ".join(open(p).read().strip().splitlines()[-n:])[-250:] if os.path.exists(p) else ""

def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT {name} ===",
         "ID  | run                     | technique            | change                         | mF1@thr     | mF1 per-label (cross-fit) | micro  | subset | lab/doc | steps     | trainable | min  | tok/s  | mem   | ms/doc"]
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            why = "SKIPPED/DROPPED" if not os.path.exists(f"{W}/logs/{e['tag']}.txt") else "FAILED"
            L.append(f"{e['id']} | {e['tag']:23s} | {why}: {log_tail(e['tag']) or log_tail('smoke_' + e['tag'])}")
            continue
        s = json.load(open(p)); dv = s["dev"]; g = dv["global"]
        L.append(f"{e['id']} | {e['tag']:23s} | {e['tech'][:20]:20s} | {e['change'][:30]:30s} | {g['macro_f1']:.4f}@{dv['global_threshold']:.2f} | "
                 f"{dv['per_label']['macro_f1']:.4f} ({dv['crossfit']['per_label']:.4f})          | {g['micro_f1']:.4f} | "
                 f"{g['subset_acc']:.4f} | {g['pred_per_doc']:5.2f}   | {s.get('best_step', 0):4d}/{s.get('steps_planned', 0):4d} | "
                 f"{s['params_trainable'] / 1e6:8.2f}M | {s['train_minutes']:4.0f} | {s.get('tok_s') or 0:6,.0f} | "
                 f"{s.get('peak_mem_gb') or 0:4.1f}G | {s.get('infer_ms_per_doc', 0):6.1f}")
        if os.path.exists(f"{RUNS}/{e['tag']}/submission.zip"):
            shutil.copy(f"{RUNS}/{e['tag']}/submission.zip", f"{W}/submissions/{e['tag']}.zip")
    bundle = f"{W}/bundle_p5_{name}.zip"              # everything except model weights (those stay in models/)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
            if os.path.exists(f"{W}/logs/{e['tag']}.txt"):
                z.write(f"{W}/logs/{e['tag']}.txt", arcname=f"{e['tag']}/stdout.txt")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("models: " + ", ".join(f"{f} ({os.path.getsize(f'{MODELS}/{f}') / 2**20:.0f} MB)" for f in sorted(os.listdir(MODELS))))
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_p5_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT 4 ===
ID  | run                     | technique            | change                         | mF1@thr     | mF1 per-label (cross-fit) | micro  | subset | lab/doc | steps     | trainable | min  | tok/s  | mem   | ms/doc
D14 | D14_s_spread8           | LoRMA r16            | whole document up to 4,096 tok | 0.2883@0.12 | 0.3360 (0.3222)          | 0.4368 | 0.0078 |  7.38   | 1500/1500 |     3.56M |   59 |  5,418 |  6.0G |   66.7
D16 | D16_s_chunkmax          | LoRMA r16            | chunk predictions + max aggreg | 0.2878@0.12 | 0.3352 (0.3197)          | 0.4362 | 0.0092 |  7.21   | 1500/1500 |     3.56M |   44 |  5,153 |  6.0G |   50.1
D18 | D18_s_bcepw             | LoRMA r16            | BCE + positive weights         | 0.3168@0.45 | 0.3434 (0.3269)          | 0.4341 | 0.0182 |  5.53   | 1500/1500 |     3.56M |   44 |  5,177 |  6.0G |   50.0
D20 | D20_s_focal             | LoRMA r16            | focal loss                     | 0.3006@0.32 | 0.3433 (0.3280)          | 0.430